# Report 2025 with DuckDB

*Data Analysis in Python · Topic 13.3*

Your team wants 2025's events by name, and 2025's purchases and revenue by device, from the event file instead of BigQuery. Query the file with DuckDB, and join the orders for the revenue.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

1. Run the parameters cell.
2. In the first cell marked ✏️, create the `events_2025` table with DuckDB: from `EVENTS`, the number of events of each `event_name` from `START_DATE` to `END_DATE`, in a column called `events`, sorted largest first.
3. In the second cell marked ✏️, create the `devices_2025` table with DuckDB: the `purchase` events from `START_DATE` to `END_DATE`, joined to `ORDERS` on `transaction_id` and `order_id`, with one row per `device_category`. Name the columns `device`, `purchases` (the number of purchases) and `revenue` (the sum of the orders' `subtotal`, rounded to 2 decimals), and sort by purchases, largest first.
4. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output.

**Expected output**: The `events_2025` table has 8 event names and 8,424,290 events, from 3,936,477 `page_view` events to 38,502 purchases. The `devices_2025` table has 3 devices, with 38,502 purchases and $3,407,991.38 of revenue, the same as topic 13.1.'s BigQuery total. Mobile has 19,039 purchases.

## Parameters

In [ ]:
import duckdb
import pandas as pd

DATA_URL = "https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/"
EVENTS = DATA_URL + "events.parquet"  # 15.1 million events, 2024 and 2025
ORDERS = DATA_URL + "orders.csv"  # One row per order, with its subtotal
START_DATE, END_DATE = "2025-01-01", "2025-12-31"

## Events by name

In [ ]:
sql = f"""
SELECT event_name, COUNT(*) AS events
FROM '{EVENTS}'
WHERE event_date BETWEEN '{START_DATE}' AND '{END_DATE}'
GROUP BY event_name
ORDER BY events DESC
"""
events_2025 = duckdb.sql(sql).df()
events_2025

## Purchases by device

In [ ]:
sql = f"""
SELECT e.device_category AS device, COUNT(*) AS purchases, ROUND(SUM(o.subtotal), 2) AS revenue
FROM '{EVENTS}' AS e
JOIN '{ORDERS}' AS o ON e.transaction_id = o.order_id
WHERE e.event_name = 'purchase' AND e.event_date BETWEEN '{START_DATE}' AND '{END_DATE}'
GROUP BY device
ORDER BY purchases DESC
"""
devices_2025 = duckdb.sql(sql).df()
devices_2025

In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables and the cells that have run. It doesn't change anything.
def check_your_work():
    import sys
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    elif checked_before:
        results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
    elif [source for source in history if source.strip() and history.count(source) > 1]:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        first_line = repeated[0].strip().splitlines()[0]
        results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
    elif error is not None:
        results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
    else:
        results.append((True, "Every cell in this session ran once, with no errors."))

    events = found.get("events_2025")
    if not isinstance(events, pd.DataFrame) or [column for column in ["event_name", "events"] if column not in events.columns]:
        results.append((False, "events_2025 should be a DataFrame with the columns event_name and events: duckdb.sql(sql).df()."))
    elif int(events["events"].sum()) == 15101655:
        results.append((False, "events_2025 counts both years' 15,101,655 events. Keep 2025's: WHERE event_date BETWEEN START_DATE AND END_DATE."))
    elif len(events) != 8 or int(events["events"].sum()) != 8424290:
        results.append((False, f"2025 has 8 event names and 8,424,290 events. events_2025 has {len(events)} and {int(events['events'].sum()):,}. Check the dates and GROUP BY event_name."))
    elif events.iloc[0]["event_name"] != "page_view":
        results.append((False, "Sort by the number of events, largest first: ORDER BY events DESC."))
    else:
        results.append((True, "8 event names and 8,424,290 events in 2025, from 3,936,477 page views to 38,502 purchases."))

    devices = found.get("devices_2025")
    columns = ["device", "purchases", "revenue"]
    if not isinstance(devices, pd.DataFrame) or [column for column in columns if column not in devices.columns]:
        results.append((False, f"devices_2025 should be a DataFrame with the columns {', '.join(columns)}."))
    elif len(devices) != 3:
        results.append((False, f"The devices_2025 table should have 3 rows, one per device. It has {len(devices)}."))
    else:
        purchases, revenue = int(devices["purchases"].sum()), float(devices["revenue"].sum())
        if purchases > 100000:
            results.append((False, f"The table counts {purchases:,} events, not purchases. Keep only the purchases: e.event_name = 'purchase'."))
        elif purchases != 38502:
            results.append((False, f"2025 has 38,502 purchases. The table has {purchases:,}. Check the dates in WHERE."))
        elif abs(revenue - 3791515.13) < 0.05:
            results.append((False, "The revenue is $3,791,515.13: the orders' total, with shipping and tax. GA4's purchase revenue is the subtotal: SUM(o.subtotal)."))
        elif abs(revenue - 3407991.38) > 0.05:
            results.append((False, f"2025's purchase revenue is $3,407,991.38. The table has ${revenue:,.2f}. Sum the orders' subtotal, joined ON e.transaction_id = o.order_id."))
        elif devices.iloc[0]["device"] != "mobile":
            results.append((False, "Sort by purchases, largest first: ORDER BY purchases DESC."))
        else:
            results.append((True, "3 devices, with 38,502 purchases and $3,407,991.38 of revenue: the same as topic 13.1.'s BigQuery total."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run.")


check_your_work()

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
sql = f"""
SELECT event_name, COUNT(*) AS events
FROM '{EVENTS}'
WHERE event_date BETWEEN '{START_DATE}' AND '{END_DATE}'
GROUP BY event_name
ORDER BY events DESC
"""
events_2025 = duckdb.sql(sql).df()
events_2025
```

```python
sql = f"""
SELECT e.device_category AS device, COUNT(*) AS purchases, ROUND(SUM(o.subtotal), 2) AS revenue
FROM '{EVENTS}' AS e
JOIN '{ORDERS}' AS o ON e.transaction_id = o.order_id
WHERE e.event_name = 'purchase' AND e.event_date BETWEEN '{START_DATE}' AND '{END_DATE}'
GROUP BY device
ORDER BY purchases DESC
"""
devices_2025 = duckdb.sql(sql).df()
devices_2025
```

</details>